# Jaguar Re-identification Training Pipeline

This notebook demonstrates the complete re-identification pipeline:
1. Configuration setup
2. Data loading from FiftyOne/disk/HuggingFace
3. Model training with backbone feature extraction
4. Evaluation
5. Results export

Based on `jaguars.reidentification.pipeline`

## Setup and Imports

In [1]:
import sys
from pathlib import Path
import logging

# Add src to path if needed
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import ReidentificationConfig, get_default_config
from jaguars.reidentification.training.train import run_processing as run_training
from jaguars.reidentification.evaluation.evaluation import run_processing as run_evaluation
from jaguars.reidentification.export_results import run_processing as run_export

# Setup logging
logger = setup_logger("reidentification_notebook", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Set Random Seeds for Reproducibility

In [2]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"✓ Random seeds set to {SEED} (GPU available)")
else:
    print(f"✓ Random seeds set to {SEED} (CPU only)")

✓ Random seeds set to 42 (GPU available)


## Configuration Setup

Configure the pipeline parameters. You can modify these based on your needs.

In [ ]:
# Get default configuration
config = get_default_config()

# Dataset Configuration
config.dataset.source = "fiftyone"  # Options: "fiftyone", "disk", "huggingface"
config.dataset.fo_dataset_name = "JID_Master_Dataset"  # FiftyOne dataset name
# config.dataset.data_dir = Path("data/jaguars")  # For disk source
# config.dataset.hf_repo = "username/jaguar-dataset"  # For HuggingFace source

# For loading from disk (exported variants):
# config.dataset.source = "disk"
# config.dataset.data_dir = Path("../camera-trap-footage/data/intermediate/v1/fo_jaguars/exports/segmented_deduplicated")

# Training Configuration
config.training.batch_size = 32
config.training.num_epochs = 50
config.training.learning_rate = 1e-4
config.training.save_dir = Path("data/models/reidentification")

# Backbone Configuration
config.backbone.name = "vit_large_patch14_dinov2.lvd142m"  # DINOv2 backbone
config.backbone.pretrained = True
config.backbone.freeze = True  # Freeze backbone during training

# Model Configuration
config.model.embedding_dim = 256
config.model.hidden_dim = 512

# Wandb Configuration (optional)
config.wandb.enabled = False  # Set to True to enable wandb logging
config.wandb.project = "jaguar-reidentification"
config.wandb.run_name = "training_run"  # Optional custom run name
# config.wandb.entity = "your-wandb-entity"  # Optional

# Evaluation Configuration
config.evaluation.save_embeddings = True
config.evaluation.save_predictions = True
config.evaluation.add_to_fiftyone = False  # Add results back to FiftyOne
config.evaluation.output_dir = Path("data/results/reidentification")

# Runtime
config.verbose = True
config.seed = SEED

print("Configuration:")
print(f"  Dataset source: {config.dataset.source}")
if config.dataset.source == "fiftyone":
    print(f"  FiftyOne dataset: {config.dataset.fo_dataset_name}")
elif config.dataset.source == "disk":
    print(f"  Data directory: {config.dataset.data_dir}")
print(f"  Backbone: {config.backbone.name}")
print(f"  Batch size: {config.training.batch_size}")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Learning rate: {config.training.learning_rate}")
print(f"  Embedding dim: {config.model.embedding_dim}")
print(f"  Wandb enabled: {config.wandb.enabled}")

Configuration:
  Dataset source: fiftyone
  Backbone: vit_large_patch14_dinov2.lvd142m
  Batch size: 32
  Epochs: 50
  Learning rate: 0.0001
  Embedding dim: 256
  Wandb enabled: False


## Load Dataset from Disk (Optional)

If you want to load a specific dataset variant from disk into FiftyOne, run this cell.

In [ ]:
# Configuration for loading from disk
LOAD_FROM_DISK = False  # Set to True to load from disk
DATASET_VARIANT = "segmented_deduplicated"  # Options: master, segmented_deduplicated, segmented, not_segmented_deduplicated, not_segmented
DATASET_DISK_PATH = Path("../camera-trap-footage/data/intermediate/v1/fo_jaguars/exports") / DATASET_VARIANT
IMPORTED_DATASET_NAME = f"JID_{DATASET_VARIANT}"
OVERWRITE_IF_EXISTS = False

if LOAD_FROM_DISK:
    import fiftyone as fo
    
    print("=" * 70)
    print(f"Loading dataset variant '{DATASET_VARIANT}' from disk")
    print("=" * 70)
    
    # Check if dataset already exists
    if fo.dataset_exists(IMPORTED_DATASET_NAME):
        if OVERWRITE_IF_EXISTS:
            print(f"Deleting existing dataset '{IMPORTED_DATASET_NAME}'...")
            fo.delete_dataset(IMPORTED_DATASET_NAME)
        else:
            print(f"✓ Dataset '{IMPORTED_DATASET_NAME}' already exists in FiftyOne")
            print("Set OVERWRITE_IF_EXISTS=True to reload from disk")
            imported_dataset = fo.load_dataset(IMPORTED_DATASET_NAME)
            print(f"  Total samples: {len(imported_dataset)}")
    
    if not fo.dataset_exists(IMPORTED_DATASET_NAME):
        if not DATASET_DISK_PATH.exists():
            print(f"⚠ Error: Dataset not found at {DATASET_DISK_PATH}")
            print("Make sure you've run the export step in the ingestion pipeline first.")
        else:
            print(f"Loading from: {DATASET_DISK_PATH}")
            imported_dataset = fo.Dataset.from_dir(
                dataset_dir=str(DATASET_DISK_PATH),
                dataset_type=fo.types.FiftyOneDataset,
                name=IMPORTED_DATASET_NAME,
            )
            print(f"✓ Dataset loaded and saved as '{IMPORTED_DATASET_NAME}'")
            print(f"  Total samples: {len(imported_dataset)}")
            
            # Update config to use this dataset
            config.dataset.source = "fiftyone"
            config.dataset.fo_dataset_name = IMPORTED_DATASET_NAME
            print(f"\n✓ Configuration updated to use '{IMPORTED_DATASET_NAME}'")
else:
    print("Skipping disk import. Set LOAD_FROM_DISK=True to import a dataset variant.")

## Step 1: Training

Train the re-identification model with the configured parameters.

In [ ]:
print("=" * 70)
print("STEP 1: Training")
print("=" * 70)

training_results = run_training(config=config, verbose=config.verbose)

print("\nTraining completed!")
print(f"Best validation mAP: {training_results.get('best_val_map', 'N/A'):.4f}")
print(f"Best model saved to: {config.training.save_dir / 'best_model.pt'}")

STEP 1: Training
21:01:20 - jid_logger.reidentification.training - INFO - Starting re-identification training...
21:01:20 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
21:01:20 - jid_logger.reidentification.training - INFO - Backbone: vit_large_patch14_dinov2.lvd142m
21:01:20 - jid_logger.reidentification.training - INFO - Device: cuda


ValueError: FiftyOne dataset 'JID_Master_Dataset' does not exist

{"t":{"$date":"2026-01-31T20:01:33.657Z"},"s":"I",  "c":"CONTROL",  "id":20697,   "ctx":"main","msg":"Renamed existing log file","attr":{"oldLogPath":"/sc/home/philipp.kolbe/.fiftyone/var/lib/mongo/log/mongo.log","newLogPath":"/sc/home/philipp.kolbe/.fiftyone/var/lib/mongo/log/mongo.log.2026-01-31T20-01-33"}}


Subprocess ['/sc/home/philipp.kolbe/conda3/envs/jid/lib/python3.14/site-packages/fiftyone/db/bin/mongod', '--dbpath', '/sc/home/philipp.kolbe/.fiftyone/var/lib/mongo', '--logpath', '/sc/home/philipp.kolbe/.fiftyone/var/lib/mongo/log/mongo.log', '--port', '0', '--nounixsocket'] exited with error -9:


## Step 2: Evaluation

Evaluate the trained model on the test set.

In [ ]:
print("=" * 70)
print("STEP 2: Evaluation")
print("=" * 70)

# Use the best model from training
model_path = config.training.save_dir / "best_model.pt"
print(f"Loading model from: {model_path}")

evaluation_results = run_evaluation(
    config=config,
    model_path=model_path,
    verbose=config.verbose
)

print("\nEvaluation completed!")
print(f"Test mAP: {evaluation_results.get('map', 'N/A'):.4f}")
if 'top1_accuracy' in evaluation_results:
    print(f"Top-1 Accuracy: {evaluation_results['top1_accuracy']:.4f}")
if 'top5_accuracy' in evaluation_results:
    print(f"Top-5 Accuracy: {evaluation_results['top5_accuracy']:.4f}")

## Step 3: Export Results

Export embeddings and predictions to disk and/or FiftyOne.

In [ ]:
print("=" * 70)
print("STEP 3: Export Results")
print("=" * 70)

# Determine export targets based on configuration
export_targets = []
if config.evaluation.save_embeddings or config.evaluation.save_predictions:
    export_targets.append("disk")
if config.evaluation.add_to_fiftyone:
    export_targets.append("fiftyone")

if not export_targets:
    print("No export targets configured. Skipping export.")
else:
    print(f"Export targets: {export_targets}")
    
    export_results = run_export(
        config=config,
        model_path=model_path,
        export_targets=export_targets,
        verbose=config.verbose
    )
    
    print("\nExport completed!")
    if "disk" in export_targets:
        print(f"Results saved to: {config.evaluation.output_dir}")
    if "fiftyone" in export_targets:
        print(f"Results added to FiftyOne dataset: {config.dataset.fo_dataset_name}")

## Summary

Display final results from all pipeline steps.

In [ ]:
print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)

print("\nConfiguration:")
print(f"  Dataset: {config.dataset.source}")
print(f"  Backbone: {config.backbone.name}")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Batch size: {config.training.batch_size}")

print("\nResults:")
if training_results:
    print(f"  Best validation mAP: {training_results.get('best_val_map', 'N/A'):.4f}")
if evaluation_results:
    print(f"  Test mAP: {evaluation_results.get('map', 'N/A'):.4f}")
    if 'top1_accuracy' in evaluation_results:
        print(f"  Top-1 Accuracy: {evaluation_results['top1_accuracy']:.4f}")
    if 'top5_accuracy' in evaluation_results:
        print(f"  Top-5 Accuracy: {evaluation_results['top5_accuracy']:.4f}")

print("\nOutput locations:")
print(f"  Model: {config.training.save_dir / 'best_model.pt'}")
if config.evaluation.save_embeddings or config.evaluation.save_predictions:
    print(f"  Results: {config.evaluation.output_dir}")

print("\n" + "=" * 70)
print("Pipeline completed successfully!")
print("=" * 70)

## Optional: Visualize Results

Load and visualize embeddings or predictions (if saved).

In [ ]:
# Example: Load and visualize embeddings with t-SNE/UMAP
# This is optional and requires the embeddings to be saved

if config.evaluation.save_embeddings:
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE
    
    # Load embeddings (adjust path as needed)
    embeddings_path = config.evaluation.output_dir / "embeddings.npz"
    
    if embeddings_path.exists():
        print(f"Loading embeddings from {embeddings_path}")
        data = np.load(embeddings_path, allow_pickle=True)
        embeddings = data['embeddings']
        labels = data['labels']
        
        print(f"Embeddings shape: {embeddings.shape}")
        print(f"Number of unique individuals: {len(np.unique(labels))}")
        
        # Reduce dimensionality for visualization
        print("Computing t-SNE projection...")
        tsne = TSNE(n_components=2, random_state=SEED)
        embeddings_2d = tsne.fit_transform(embeddings)
        
        # Plot
        plt.figure(figsize=(12, 8))
        scatter = plt.scatter(
            embeddings_2d[:, 0],
            embeddings_2d[:, 1],
            c=labels,
            cmap='tab20',
            alpha=0.6
        )
        plt.colorbar(scatter, label='Individual ID')
        plt.title('Jaguar Re-identification Embeddings (t-SNE)')
        plt.xlabel('t-SNE 1')
        plt.ylabel('t-SNE 2')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Embeddings file not found at {embeddings_path}")
else:
    print("Embeddings were not saved. Set config.evaluation.save_embeddings = True to enable visualization.")

## Optional: Load Model for Inference

Load the trained model for making predictions on new data.

In [ ]:
# Example: Load model for inference
import torch
from jaguars.reidentification.model import ReidentificationModel
from jaguars.reidentification.backbone import get_backbone

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize backbone and model
backbone = get_backbone(
    name=config.backbone.name,
    pretrained=config.backbone.pretrained
)

model = ReidentificationModel(
    backbone=backbone,
    embedding_dim=config.model.embedding_dim,
    hidden_dim=config.model.hidden_dim
)

# Load weights
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"✓ Model loaded from {model_path}")
print(f"  Trained for {checkpoint.get('epoch', 'N/A')} epochs")
print(f"  Best validation mAP: {checkpoint.get('best_map', 'N/A'):.4f}")

# Now you can use the model for inference
# Example:
# with torch.no_grad():
#     embeddings = model(batch_images)
#     # Compare with gallery embeddings using cosine similarity